# Tafawoq — TassiliTTS test (Google Colab)

Runs the Algerian Dardja voice model (https://huggingface.co/samifalouti/TassiliTTS) on a free Colab GPU to **listen** and **measure** — not to host the platform (a Colab session stops after a few hours and is not a commercial host).

1. Runtime → Change runtime type → **T4 GPU**.
2. Run the cells in order.
3. Upload a **reference recording** only if its speaker gave written consent (or use the synthetic espeak reference the notebook can create).

Licence: the model card says MIT, but the base model (Habibi-TTS) and the training data must be checked before any commercial use — see `services/tassili-tts/README.md`.

In [ ]:
!nvidia-smi -L || echo 'No GPU: inference will be slow on CPU'
!pip -q install habibi-tts==0.1.1 soundfile
!apt-get -qq install -y espeak-ng > /dev/null

In [ ]:
from huggingface_hub import hf_hub_download
ckpt = hf_hub_download('samifalouti/TassiliTTS', 'ALGERIA.safetensors')
vocab = hf_hub_download('samifalouti/TassiliTTS', 'vocab.txt')
print(ckpt, vocab)

In [ ]:
import time, torch
from f5_tts.infer.utils_infer import load_model, load_vocoder, preprocess_ref_audio_text
from f5_tts.model import DiT
device = 'cuda' if torch.cuda.is_available() else 'cpu'
t0 = time.time()
vocoder = load_vocoder(vocoder_name='vocos', is_local=False, device=device)
model = load_model(DiT, dict(dim=1024, depth=22, heads=16, ff_mult=2, text_dim=512, conv_layers=4),
                   ckpt_path=ckpt, vocab_file=vocab, device=device)
print(f'loaded on {device} in {time.time()-t0:.1f}s')

## Reference voice
Default: a **synthetic** espeak-ng reference (no real person — it sounds robotic, and the clone will too). To hear a real teacher voice, upload `reference.wav` (5–15 s, clean) recorded by someone who signed a consent, and put its exact transcript in `REF_TEXT`.

In [ ]:
import os
REF_AUDIO = 'reference.wav'
REF_TEXT = 'السلام عليكم، اليوم نراجعو الدرس مع بعض'
if not os.path.exists(REF_AUDIO):
    !espeak-ng -v ar -s 140 -w reference.wav "السلام عليكم، اليوم نراجعو الدرس مع بعض"
ref_audio, ref_text = preprocess_ref_audio_text(REF_AUDIO, REF_TEXT)

In [ ]:
import soundfile as sf
from IPython.display import Audio, display
from habibi_tts.infer.utils_infer import infer_process
# Arabic script only, numbers written in words (the model cannot read digits).
sentences = [
    'واش راك خويا، اليوم نشوفو المشتقة مع بعض',
    'إف مشتقة إكس يساوي اثنان إكس زائد ثلاثة',
    'صحيت، هاديك هي الإجابة الصحيحة',
]
for i, text in enumerate(sentences):
    t0 = time.time()
    audio, sr, _ = infer_process(ref_audio=ref_audio, ref_text=ref_text, gen_text=text,
                                 model_obj=model, vocoder=vocoder, speed=1.0, nfe_step=32, device=device)
    took = time.time() - t0
    sf.write(f'out_{i}.wav', audio, sr)
    print(f'{text}\n  {took:.1f}s to make {len(audio)/sr:.1f}s of audio (real-time factor {took/(len(audio)/sr):.2f})')
    display(Audio(f'out_{i}.wav'))

## Optional: run the Tafawoq service here for a short test
Upload `services/tassili-tts/app.py`, write `voices.json`, then start it and expose it with a tunnel of your choice (cloudflared, ngrok). Set a long random `TASSILI_TOKEN` and give the URL and token to the Node server as `TAFAWOQ_TASSILI_URL` / `TAFAWOQ_TASSILI_TOKEN`. The URL changes at every session: this is for trying, not hosting.

In [ ]:
# import json, secrets, subprocess
# json.dump({'male': {'ref_audio': 'reference.wav', 'ref_text': REF_TEXT, 'consent_ref': 'synthetic espeak-ng voice, no person'}}, open('voices.json', 'w'), ensure_ascii=False)
# token = secrets.token_urlsafe(32); print('TASSILI_TOKEN =', token)
# subprocess.Popen(f'TASSILI_TOKEN={token} TASSILI_CKPT={ckpt} TASSILI_VOCAB={vocab} TASSILI_VOICES=voices.json uvicorn app:app --port 8008', shell=True)